# 1 – Indexing: Dokumente in eine Vektordatenbank + BM25-Index überführen

Dieses Notebook führt die **Indexing-Pipeline** durch:

1. **Laden** – PDFs und Word-Dokumente werden eingelesen
2. **Chunking** – Der Text wird in überlappende Abschnitte zerlegt
3. **Embedding** – Jeder Chunk wird in einen Vektor umgewandelt → ChromaDB
4. **BM25-Index** – Die gleichen Chunks werden für lexikalische Suche gespeichert

> **Neu in dieser Version:** Neben der Vektordatenbank wird ein **zweiter, lexikalischer Index**
> (BM25) erstellt. Beide Indizes basieren auf denselben Chunks, nutzen aber
> grundlegend unterschiedliche Suchverfahren – das ist die Basis für *Hybrid Search*.

> **Hinweis:** Dieses Notebook muss nur einmal (oder bei neuen Dokumenten) ausgeführt werden.  
> Das RAG-Notebook (Notebook 3) liest dann beide Indizes.

## Imports & Konfiguration

In [1]:
import os
import glob
import shutil
import pickle

from langchain_community.document_loaders import PyMuPDFLoader, Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

/Users/thomasjorg/Documents/05_VSCode/05_Python/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# --- KONFIGURATION ---

# Pfade
PDF_SOURCE_DIR = "./documents"      # Ordner mit den Quell-Dokumenten (PDF + DOCX)
DB_DIR         = "./chroma_db"       # Speicherort der Vektordatenbank
BM25_DIR       = "./bm25_index"      # NEU: Speicherort für den BM25-Index
MODEL_PATH     = "./models"          # Lokaler Cache für das Embedding-Modell

# Embedding-Modell
EMBEDDING_MODEL = "intfloat/multilingual-e5-large-instruct"

# Chunking-Parameter
CHUNK_SIZE    = 512
CHUNK_OVERLAP = 128   # ~25 % Überlappung – wichtig bei langen deutschen Sätzen

## Embedding-Modell vorbereiten

Das Modell `multilingual-e5-large-instruct` erwartet **spezifische Präfixe**:  
- Dokumente (beim Indexieren): `"passage: ..."`  
- Suchanfragen (beim Retrieval): `"query: ..."`  

Ohne diese Präfixe arbeitet das Modell deutlich unter seinem Niveau.  
Da LangChain diese Präfixe nicht automatisch setzt, bauen wir einen dünnen Wrapper:

In [3]:
class E5Embeddings(HuggingFaceEmbeddings):
    """Wrapper, der die vom E5-Modell erwarteten Präfixe automatisch setzt."""

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return super().embed_documents(["passage: " + t for t in texts])

    def embed_query(self, text: str) -> list[float]:
        return super().embed_query("query: " + text)


embeddings = E5Embeddings(
    model_name=EMBEDDING_MODEL,
    cache_folder=MODEL_PATH,
)

## Schritt 1 – Dokumente laden (PDF & DOCX)

In [4]:
# Mapping: Dateiendung → passender LangChain-Loader
LOADERS = {
    ".pdf":  PyMuPDFLoader,
    ".docx": Docx2txtLoader,
}


def load_documents(source_dir: str):
    """Lädt alle unterstützten Dokumente (PDF, DOCX) aus einem Verzeichnis."""
    all_docs = []
    file_count = 0

    for ext, loader_cls in LOADERS.items():
        for filepath in glob.glob(os.path.join(source_dir, f"*{ext}")):
            loader = loader_cls(filepath)
            all_docs.extend(loader.load())
            print(f"  📄 {os.path.basename(filepath)}")
            file_count += 1

    print(f"\n✅ {len(all_docs)} Seiten/Abschnitte aus {file_count} Dokumenten geladen.")
    return all_docs

## Schritt 2 – Text in Chunks aufteilen

In [5]:
def split_documents(docs, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP):
    """Zerlegt Dokumente in überlappende Text-Chunks."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    chunks = splitter.split_documents(docs)
    print(f"✅ {len(chunks)} Chunks erzeugt (Größe: {chunk_size}, Overlap: {chunk_overlap})")
    return chunks

## Schritt 3 – Embeddings erstellen & in ChromaDB speichern

In [6]:
def create_vectorstore(chunks, embedding_fn, db_dir=DB_DIR):
    """Erzeugt Embeddings und speichert sie in einer lokalen ChromaDB."""
    # Alten Index löschen, damit keine veralteten Vektoren drin bleiben
    if os.path.exists(db_dir):
        shutil.rmtree(db_dir)
        print("🗑️  Alter Vektor-Index gelöscht.")

    print("🧠 Erstelle Embeddings und speichere in ChromaDB...")
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_fn,
        persist_directory=db_dir,
    )
    print(f"✨ Vektor-Index gespeichert in '{db_dir}'")
    return vectorstore

## Schritt 4 – BM25-Index erstellen & persistieren

BM25 ist ein **lexikalisches** Suchverfahren – es arbeitet mit Wort-Häufigkeiten
statt mit Vektoren. Damit ergänzt es die semantische Suche ideal:

| | Vektorsuche (E5) | Lexikalische Suche (BM25) |
|---|---|---|
| **Stärke** | Paraphrasen, sinngemäße Fragen | Exakte Begriffe, Fachtermini, Abkürzungen |
| **Schwäche** | Exakte Schlüsselwörter gehen unter | Versteht keine Synonyme |

Wir speichern die Chunks als Pickle-Datei, damit der BM25-Retriever
im RAG-Notebook sofort geladen werden kann, ohne die Dokumente erneut zu parsen.

In [7]:
def create_bm25_index(chunks, bm25_dir=BM25_DIR):
    """Speichert die Chunks für den BM25-Retriever als Pickle-Datei.
    
    Der BM25-Retriever von LangChain baut seinen Index zur Laufzeit
    aus LangChain-Documents auf. Wir müssen also nur die Documents
    persistieren – den eigentlichen BM25-Index berechnet der Retriever
    beim Laden automatisch (das dauert nur Sekundenbruchteile).
    """
    os.makedirs(bm25_dir, exist_ok=True)
    
    index_path = os.path.join(bm25_dir, "chunks.pkl")
    
    # Alten Index löschen
    if os.path.exists(index_path):
        os.remove(index_path)
        print("🗑️  Alter BM25-Index gelöscht.")
    
    with open(index_path, "wb") as f:
        pickle.dump(chunks, f)
    
    print(f"✨ BM25-Index gespeichert in '{index_path}' ({len(chunks)} Chunks)")
    return index_path

## Pipeline ausführen

In [8]:
# --- PIPELINE ---
print("🚀 Starte Indexing-Pipeline...\n")

docs   = load_documents(PDF_SOURCE_DIR)
chunks = split_documents(docs)

# Index 1: Semantisch (Vektordatenbank)
print("\n--- Index 1: Vektordatenbank (ChromaDB) ---")
db = create_vectorstore(chunks, embeddings)

# Index 2: Lexikalisch (BM25)
print("\n--- Index 2: Lexikalischer Index (BM25) ---")
bm25_path = create_bm25_index(chunks)

print(f"\n🎯 Beide Indizes erstellt! Nächster Schritt: Notebook 3 (RAG) öffnen.")
print(f"   📁 Vektor-Index:  {DB_DIR}")
print(f"   📁 BM25-Index:    {BM25_DIR}")

🚀 Starte Indexing-Pipeline...

  📄 Physik_2016_Loesungsvorschlag.pdf
  📄 Physik_2020-Loesungsvorschlag.pdf
  📄 Physik_2019-Loesungshinweise.pdf
  📄 Physik_2009.pdf
  📄 Physik_2014_Loesungsvorschlag.pdf
  📄 Physik_2008.pdf
  📄 Physik_2018-Loesungshinweise.pdf
  📄 Physik_2012_Loesungsvorschlag.pdf
  📄 Physik_2010_Loesungsvorschlag.pdf
  📄 2025_Phy_eAN_Aufgaben_V.pdf
  📄 Physik_2009_Loesungsvorschlag.pdf
  📄 2023_Phy_eAN_Aufgaben_2023_09_21.pdf
  📄 Physik 2021 - Aufgaben.pdf
  📄 Physik 2023 - Lösungshinweise.pdf
  📄 Physik 2020 - Aufgaben.pdf
  📄 2024_Phy_eAN_Aufgaben_V_.pdf
  📄 Physik_2015_Loesungsvorschlag.pdf
  📄 Physik_2012.pdf
  📄 Physik_2006.pdf
  📄 Physik 2022 - Aufgaben.pdf
  📄 Physik_2013_Loesungsvorschlag.pdf
  📄 Physik 2020 - Lösungsvorschlag.pdf
  📄 Physik_2007.pdf
  📄 Physik_2013.pdf
  📄 Physik 2025 - Aufgaben.pdf
  📄 Physik_2005.pdf
  📄 Physik_2011.pdf
  📄 Physik_2018-Aufgaben.pdf
  📄 Physik_2010.pdf
  📄 Physik_2004.pdf
  📄 Physik_2014.pdf
  📄 Physik 2024 - Aufgaben.pdf
  

In [9]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"MKL verfügbar: {torch.backends.mkl.is_available()}")
print(f"Threads: {torch.get_num_threads()}")

PyTorch: 2.8.0
MKL verfügbar: False
Threads: 8
